In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# YF 쏘나타 검색 결과 URL
url = "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7"
}

resp = requests.get(url, headers=headers, timeout=15)
resp.raise_for_status()

print(resp.status_code)
print(resp.text[:1000])   # 먼저 구조 확인

200
<!DOCTYPE html><html lang="ko"><head><meta charSet="utf-8"/><meta name="naver-site-verification" content="d6816425de244fa5871cff7ed7aa527a3ec6c481"/><meta name="google-site-verification" content="k-XSUZ77xZ_-SZbScyPs2ahKnh-FPiib6Ks4Qoll-Q0"/><meta name="facebook-domain-verification" content="2xnfhiu5wheuiglzcp22q1c2gjsbpf"/><link rel="shortcut icon" href="/favicon.ico"/><meta http-equiv="cache-control" content="no-cache"/><meta http-equiv="expires" content="Tue, 01 Jan 1980 1:00:00 GMT"/><meta http-equiv="pragma" content="no-cache"/><link href="https://ci.encar.com" rel="dns-prefetch"/><link href="https://static.encar.com" rel="dns-prefetch"/><title>엔카믿고 현대 YF 쏘나타 중고차 : 내차팔기·내차사기</title><meta property="og:title" content="엔카믿고 현대 YF 쏘나타 중고차 : 내차팔기·내차사기"/><meta name="description" content="YF 쏘나타중고차는 역시 엔카! 최다 매물과 편리한 구매 서비스로 나만의 중고차를 찾아보세요."/><meta property="og:description" content="YF 쏘나타중고차는 역시 엔카! 최다 매물과 편리한 구매 서비스로 나만의 중고차를 찾아보세요."/><meta property="og:url" content="https://car.en

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from urllib.parse import urljoin

BASE_URL = "https://car.encar.com"

# YF 쏘나타 검색 URL
SEARCH_URL_TEMPLATE = (
    "https://car.encar.com/list/car?page={page}"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://car.encar.com/"
}

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "가솔린", "디젤", "LPG", "LPG(일반인 구입)", "하이브리드", "가솔린+전기",
    "전기", "CNG", "수소"
]


def extract_price(text: str):
    m = re.search(r'(\d[\d,]*)\s*만원', text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_mileage(text: str):
    m = re.search(r'(\d[\d,]*)\s*km', text, re.IGNORECASE)
    return int(m.group(1).replace(",", "")) if m else None


def extract_year(text: str):
    # 예: 10/10식(11년형), 12/04식
    m = re.search(r'(\d{2}/\d{2}식(?:\(\d{2}년형\))?)', text)
    return m.group(1) if m else None


def extract_fuel(text: str):
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text: str):
    for region in REGIONS:
        if region in text:
            return region
    return None


def extract_name(text: str, year_text: str):
    """
    연식 앞부분까지를 차량명으로 추정
    """
    if year_text and year_text in text:
        name = text.split(year_text)[0].strip()
        return re.sub(r'\s+', ' ', name)
    return None


def normalize_text(text: str):
    return re.sub(r'\s+', ' ', text).strip()


def collect_candidate_links(card, base_url=BASE_URL):
    links = []
    for a_tag in card.select("a[href]"):
        href = a_tag.get("href", "").strip()
        if not href:
            continue
        full_url = urljoin(base_url, href)
        links.append(full_url)

    # 중복 제거
    deduped = []
    seen = set()
    for link in links:
        if link not in seen:
            deduped.append(link)
            seen.add(link)
    return deduped


def parse_card(card):
    raw_text = normalize_text(card.get_text(" ", strip=True))

    if "YF 쏘나타" not in raw_text:
        return None

    if "만원" not in raw_text or "km" not in raw_text:
        return None

    year_text = extract_year(raw_text)
    name = extract_name(raw_text, year_text)
    price = extract_price(raw_text)
    mileage = extract_mileage(raw_text)
    fuel = extract_fuel(raw_text)
    region = extract_region(raw_text)
    links = collect_candidate_links(card)

    return {
        "차량명": name,
        "연식": year_text,
        "주행거리_km": mileage,
        "연료": fuel,
        "지역": region,
        "가격_만원": price,
        "상세링크_후보": " | ".join(links[:5]) if links else None,
        "raw_text": raw_text
    }


def fetch_page(page: int):
    url = SEARCH_URL_TEMPLATE.format(page=page)
    print(f"[INFO] 요청 중: {url}")

    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()

    return response.text


def parse_page(html: str):
    soup = BeautifulSoup(html, "html.parser")

    rows = []

    # li / div / article 전부 후보로 본다
    candidate_cards = soup.select("li, div, article")

    for card in candidate_cards:
        item = parse_card(card)
        if item:
            rows.append(item)

    # 중복 제거용
    if rows:
        df = pd.DataFrame(rows)
        df = df.drop_duplicates(subset=["차량명", "연식", "주행거리_km", "가격_만원", "raw_text"])
        return df

    return pd.DataFrame()


def scrape_yf_sonata(max_pages=5, delay=1.5, save_csv=True):
    all_dfs = []

    for page in range(1, max_pages + 1):
        try:
            html = fetch_page(page)
            df_page = parse_page(html)

            print(f"[INFO] page={page}, 수집건수={len(df_page)}")

            if df_page.empty:
                print("[INFO] 빈 페이지 또는 파싱 실패로 판단, 중단합니다.")
                break

            all_dfs.append(df_page)
            time.sleep(delay)

        except Exception as e:
            print(f"[ERROR] page={page} 수집 실패: {e}")
            break

    if not all_dfs:
        print("[WARN] 수집된 데이터가 없습니다.")
        return pd.DataFrame()

    df_final = pd.concat(all_dfs, ignore_index=True)
    df_final = df_final.drop_duplicates(subset=["차량명", "연식", "주행거리_km", "가격_만원", "raw_text"])
    df_final = df_final.reset_index(drop=True)

    if save_csv:
        filename = "encar_yf_sonata.csv"
        df_final.to_csv(filename, index=False, encoding="utf-8-sig")
        print(f"[INFO] 저장 완료: {filename}")

    return df_final


if __name__ == "__main__":
    df = scrape_yf_sonata(max_pages=10, delay=2.0, save_csv=True)
    print(df.head(20))
    print(f"\n총 수집 건수: {len(df)}")

[INFO] 요청 중: https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
[INFO] page=1, 수집건수=25
[INFO] 요청 중: https://car.encar.com/list/car?page=2&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
[INFO] page=2, 수집건수=0
[INFO] 빈 페이지 또는 파싱 실패로 판단, 중단합니다.
[INFO] 저장 완료: encar_yf_sonata.csv
                                                  차량명            연식  주행거리_km  \
0   차량검색 확장메뉴열기 최근 업데이트순 검색 조건 변경 0 대 검색조건 적용됨 확장메...  09/10식(10년형)    98

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

SEARCH_URL_TEMPLATE = (
    "https://car.encar.com/list/car?page={page}"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://car.encar.com/"
}

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "LPG(일반인 구입)", "가솔린+전기", "하이브리드", "가솔린", "디젤", "LPG", "전기"
]


def normalize_text(text: str):
    return re.sub(r"\s+", " ", text).strip()


def extract_detail_link(tag):
    a_tags = tag.select('a[href*="fem.encar.com/cars/detail/"]')
    if not a_tags:
        return None

    href = a_tags[0].get("href", "").strip()
    return href if href else None


def extract_car_id(link: str):
    if not link:
        return None
    m = re.search(r"/cars/detail/(\d+)", link)
    return m.group(1) if m else None


def extract_price(text: str):
    m = re.search(r'(\d[\d,]*)\s*만원', text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_mileage(text: str):
    m = re.search(r'(\d[\d,]*)\s*km', text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_year(text: str):
    m = re.search(r'(\d{2}/\d{2}식(?:\(\d{2}년형\))?)', text)
    return m.group(1) if m else None


def extract_fuel(text: str):
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text: str):
    for region in REGIONS:
        if region in text:
            return region
    return None


def extract_name(text: str):
    """
    차량명은 'YF 쏘나타'부터 연식 직전까지 추출
    """
    m = re.search(r'(YF\s*쏘나타.*?)(?=\s+\d{2}/\d{2}식(?:\(\d{2}년형\))?)', text)
    if m:
        return normalize_text(m.group(1))
    return None


def is_valid_row(name, price, mileage, year_text):
    if not name or "YF 쏘나타" not in name:
        return False
    if price is None or mileage is None or year_text is None:
        return False
    return True


def fetch_page(page: int):
    url = SEARCH_URL_TEMPLATE.format(page=page)
    print(f"[INFO] 요청 중: {url}")
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text


def parse_page(html: str):
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    # detail 링크가 있는 a 태그를 먼저 찾고,
    # 그 부모 블록을 후보 카드로 본다
    detail_links = soup.select('a[href*="fem.encar.com/cars/detail/"]')
    seen_ids = set()

    for a in detail_links:
        href = a.get("href", "").strip()
        car_id = extract_car_id(href)
        if not car_id or car_id in seen_ids:
            continue

        # 너무 작은 a 태그 자체가 아니라 어느 정도 상위 블록까지 올려서 텍스트 확보
        card = a
        for _ in range(4):
            if card.parent:
                card = card.parent

        raw_text = normalize_text(card.get_text(" ", strip=True))

        name = extract_name(raw_text)
        year_text = extract_year(raw_text)
        mileage = extract_mileage(raw_text)
        fuel = extract_fuel(raw_text)
        region = extract_region(raw_text)
        price = extract_price(raw_text)

        if not is_valid_row(name, price, mileage, year_text):
            continue

        rows.append({
            "매물ID": car_id,
            "차량명": name,
            "연식": year_text,
            "주행거리_km": mileage,
            "연료": fuel,
            "지역": region,
            "가격_만원": price,
            "상세링크": href,
            "raw_text": raw_text
        })
        seen_ids.add(car_id)

    return pd.DataFrame(rows)


def scrape_yf_sonata(max_pages=5, delay=1.5):
    all_dfs = []

    for page in range(1, max_pages + 1):
        try:
            html = fetch_page(page)
            df_page = parse_page(html)

            print(f"[INFO] page={page}, 수집건수={len(df_page)}")

            if df_page.empty:
                print("[INFO] 더 이상 유효한 매물이 없어서 중단합니다.")
                break

            all_dfs.append(df_page)
            time.sleep(delay)

        except Exception as e:
            print(f"[ERROR] page={page} 수집 실패: {e}")
            break

    if not all_dfs:
        return pd.DataFrame()

    df = pd.concat(all_dfs, ignore_index=True)
    df = df.drop_duplicates(subset=["매물ID"]).reset_index(drop=True)
    df.to_csv("encar_yf_sonata_clean.csv", index=False, encoding="utf-8-sig")
    print("[INFO] 저장 완료: encar_yf_sonata_clean.csv")
    return df


if __name__ == "__main__":
    df = scrape_yf_sonata(max_pages=10, delay=2.0)
    print(df.head(20))
    print(f"\n총 수집 건수: {len(df)}")

[INFO] 요청 중: https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
[INFO] page=1, 수집건수=9
[INFO] 요청 중: https://car.encar.com/list/car?page=2&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
[INFO] page=2, 수집건수=0
[INFO] 더 이상 유효한 매물이 없어서 중단합니다.
[INFO] 저장 완료: encar_yf_sonata_clean.csv
       매물ID                 차량명            연식  주행거리_km           연료  지역  \
0  39486507        YF 쏘나타 LPI 탑  10/11식(11년형)   158915  LPG(일반인 구입)  서울   
1

In [5]:
# pip install selenium pandas webdriver-manager

import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "LPG(일반인 구입)", "가솔린+전기", "하이브리드", "가솔린", "디젤", "LPG", "전기"
]


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def extract_car_id(url: str):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def extract_name(text: str):
    m = re.search(r"(YF\s*쏘나타.*?)(?=\s+\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    return normalize_text(m.group(1)) if m else None


def extract_year(text: str):
    m = re.search(r"(\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    return m.group(1) if m else None


def extract_mileage(text: str):
    m = re.search(r"(\d[\d,]*)\s*km", text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_price(text: str):
    m = re.search(r"(\d[\d,]*)\s*만원", text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_fuel(text: str):
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text: str):
    for region in REGIONS:
        if region in text:
            return region
    return None


def setup_driver():
    options = Options()
    # options.add_argument("--headless=new")  # 필요하면 주석 해제
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.implicitly_wait(3)
    return driver


def scroll_to_bottom(driver, pause=2.0, max_rounds=15):
    last_height = driver.execute_script("return document.body.scrollHeight")

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(f"[INFO] scroll round {i+1}: {last_height} -> {new_height}")

        if new_height == last_height:
            print("[INFO] 더 이상 스크롤로 로드되는 내용이 없어 보입니다.")
            break

        last_height = new_height


def collect_detail_urls(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="fem.encar.com/cars/detail/"]')

    urls = []
    for a in anchors:
        href = a.get_attribute("href")
        if href and "/cars/detail/" in href:
            urls.append(href.split("&advClickPosition=")[0])

    urls = list(dict.fromkeys(urls))
    print(f"[INFO] 상세 링크 수집: {len(urls)}건")
    return urls


def scrape_from_loaded_page(driver, detail_urls):
    rows = []

    for idx, url in enumerate(detail_urls, start=1):
        try:
            car_id = extract_car_id(url)
            print(f"[INFO] ({idx}/{len(detail_urls)}) 수집 중: {car_id}")

            driver.get(url)
            time.sleep(2)

            text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)

            name = extract_name(text)
            year_text = extract_year(text)
            mileage = extract_mileage(text)
            fuel = extract_fuel(text)
            region = extract_region(text)
            price = extract_price(text)

            if not name or not year_text or price is None or mileage is None:
                print(f"[WARN] 핵심 필드 누락, 스킵: {car_id}")
                continue

            rows.append({
                "매물ID": car_id,
                "차량명": name,
                "연식": year_text,
                "주행거리_km": mileage,
                "연료": fuel,
                "지역": region,
                "가격_만원": price,
                "상세링크": url,
                "raw_text": text[:3000]
            })

        except Exception as e:
            print(f"[ERROR] {url} 수집 실패: {e}")

    df = pd.DataFrame(rows).drop_duplicates(subset=["매물ID"]).reset_index(drop=True)
    return df


def main():
    driver = setup_driver()

    try:
        print("[INFO] 검색 페이지 접속")
        driver.get(SEARCH_URL)
        time.sleep(3)

        scroll_to_bottom(driver, pause=2.0, max_rounds=20)
        detail_urls = collect_detail_urls(driver)

        df = scrape_from_loaded_page(driver, detail_urls)
        df.to_csv("encar_yf_sonata_selenium.csv", index=False, encoding="utf-8-sig")

        print(df.head(20))
        print(f"\n총 수집 건수: {len(df)}")
        print("[INFO] 저장 완료: encar_yf_sonata_selenium.csv")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] 검색 페이지 접속
[INFO] scroll round 1: 71394 -> 71394
[INFO] 더 이상 스크롤로 로드되는 내용이 없어 보입니다.
[INFO] 상세 링크 수집: 210건
[INFO] (1/210) 수집 중: 40903390
[INFO] (2/210) 수집 중: 41509017
[INFO] (3/210) 수집 중: 40477859
[INFO] (4/210) 수집 중: 39486507
[INFO] (5/210) 수집 중: 41054615
[INFO] (6/210) 수집 중: 41225357
[INFO] (7/210) 수집 중: 40955073
[INFO] (8/210) 수집 중: 41265617
[INFO] (9/210) 수집 중: 41310694
[INFO] (10/210) 수집 중: 40955073
[INFO] (11/210) 수집 중: 41643943
[INFO] (12/210) 수집 중: 39444771
[INFO] (13/210) 수집 중: 41639147
[INFO] (14/210) 수집 중: 41019165
[INFO] (15/210) 수집 중: 41054615
[INFO] (16/210) 수집 중: 41038293
[INFO] (17/210) 수집 중: 41018137
[INFO] (18/210) 수집 중: 41225133
[INFO] (19/210) 수집 중: 41265648
[INFO] (20/210) 수집 중: 41300549
[INFO] (21/210) 수집 중: 41576521
[INFO] (22/210) 수집 중: 39501982
[INFO] (23/210) 수집 중: 40512700
[INFO] (24/210) 수집 중: 40041895
[INFO] (25/210) 수집 중: 41508073
[INFO] (26/210) 수집 중: 40412711
[INFO] (27/210) 수집 중: 39979642
[INFO] (28/210) 수집 중: 41265617
[INFO] (29/210) 수집 중: 4133899

In [6]:
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "LPG(일반인 구입)", "가솔린+전기", "하이브리드", "가솔린", "디젤", "LPG", "전기", "수소", "CNG"
]

TRANSMISSIONS = [
    "오토", "자동", "수동", "세미오토", "CVT"
]

POPULAR_COLORS = [
    "흰색", "화이트", "검정", "블랙", "쥐색", "회색", "그레이", "은색", "실버"
]


def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def extract_car_id(url: str):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def scroll_until_stable(driver, pause=2.0, max_rounds=20):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(f"[SCROLL] {i+1}회차: {last_height} -> {new_height}")

        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            print("[INFO] 스크롤 로딩 종료로 판단")
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        href = a.get_attribute("href")
        if href and "/cars/detail/" in href:
            href = href.split("&advClickPosition=")[0]
            urls.append(href)

    urls = list(dict.fromkeys(urls))
    return urls


# ---------- 텍스트 파싱 함수들 ----------

def extract_name(text: str):
    text = normalize_text(text)

    # YF 쏘나타부터 연식 전까지
    m = re.search(r"(YF\s*쏘나타.*?)(?=\s+\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    if m:
        return m.group(1).strip()

    return None


def split_model_trim(name: str):
    if not name:
        return None, None, None

    manufacturer = "현대"  # 현재는 YF 쏘나타 전용
    model = "YF 쏘나타"

    trim = name.replace("YF 쏘나타", "").strip()
    trim = trim if trim else None

    return manufacturer, model, trim


def extract_year(text: str):
    m = re.search(r"(\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    return m.group(1) if m else None


def convert_year_text(year_text: str):
    """
    10/05식 -> 2010
    12/04식 -> 2012
    """
    if not year_text:
        return None
    m = re.search(r"(\d{2})/\d{2}식", year_text)
    if not m:
        return None
    yy = int(m.group(1))
    return 2000 + yy


def extract_mileage(text: str):
    m = re.search(r"(\d[\d,]*)\s*km", text, re.IGNORECASE)
    return int(m.group(1).replace(",", "")) if m else None


def extract_price(text: str):
    # 가장 처음 등장하는 만원 단위 가격
    m = re.search(r"(\d[\d,]*)\s*만원", text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_fuel(text: str):
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text: str):
    for region in REGIONS:
        if region in text:
            return region
    return None


def extract_transmission(text: str):
    for tm in TRANSMISSIONS:
        if tm in text:
            return tm
    return None


def extract_displacement_cc(text: str):
    patterns = [
        r"(\d{3,4})\s*cc",
        r"배기량\s*(\d{3,4})",
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return int(m.group(1))
    return None


def extract_color(text: str):
    """
    아주 정교하진 않지만, 우선순위 높은 방식:
    1) '색상' 키워드 주변 탐색
    2) 주요 색상 키워드 탐색
    """
    # 예: 색상 흰색 / 외장색 검정
    m = re.search(r"(?:색상|외장색)\s*[: ]?\s*([가-힣A-Za-z]+)", text)
    if m:
        return m.group(1).strip()

    for c in POPULAR_COLORS:
        if c in text:
            return c

    return None


def extract_accident_flag(text: str):
    """
    사고 여부를 단순 규칙으로 추출
    """
    text = normalize_text(text)

    positive_no_accident = [
        "무사고",
        "완전무사고"
    ]
    negative_accident = [
        "사고",
        "교환",
        "수리"
    ]

    # 무사고가 있으면 우선 긍정
    for kw in positive_no_accident:
        if kw in text:
            return "무사고"

    # 사고 관련 문구가 있으면 사고이력 추정
    for kw in negative_accident:
        if kw in text:
            return "사고/수리이력 의심"

    return None


def extract_flood_flag(text: str):
    text = normalize_text(text)

    if "침수이력 없음" in text or "침수 없음" in text:
        return "침수없음"
    if "침수" in text:
        return "침수이력 의심"

    return None


def extract_usage_history(text: str):
    text = normalize_text(text)

    found = []
    if "렌트" in text or "렌터카" in text:
        found.append("렌트이력")
    if "영업용" in text:
        found.append("영업용이력")
    if "법인" in text:
        found.append("법인이력")

    if found:
        return ",".join(found)
    return None


def extract_owner_change_count(text: str):
    patterns = [
        r"소유자 변경\s*(\d+)\s*회",
        r"소유자변경\s*(\d+)\s*회",
        r"명의 변경\s*(\d+)\s*회"
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            return int(m.group(1))
    return None


def extract_options(text: str):
    option_keywords = [
        "선루프", "내비", "네비", "스마트키", "후방카메라",
        "열선시트", "통풍시트", "가죽시트", "메모리시트",
        "크루즈컨트롤", "스마트크루즈", "차선이탈", "어라운드뷰", "블랙박스"
    ]
    found = [kw for kw in option_keywords if kw in text]
    return ",".join(sorted(set(found))) if found else None


def parse_detail_text(text: str, url: str):
    text = normalize_text(text)

    name = extract_name(text)
    manufacturer, model, trim = split_model_trim(name)

    year_text = extract_year(text)

    row = {
        "매물ID": extract_car_id(url),
        "제조사": manufacturer,
        "모델": model,
        "세부트림": trim,
        "차량명": name,
        "연식_원문": year_text,
        "연식": convert_year_text(year_text),
        "주행거리_km": extract_mileage(text),
        "연료": extract_fuel(text),
        "변속기": extract_transmission(text),
        "배기량_cc": extract_displacement_cc(text),
        "색상": extract_color(text),
        "지역": extract_region(text),
        "가격_만원": extract_price(text),
        "사고유무": extract_accident_flag(text),
        "침수유무": extract_flood_flag(text),
        "용도이력": extract_usage_history(text),
        "소유자변경횟수": extract_owner_change_count(text),
        "옵션원문": extract_options(text),
        "상세링크": url,
        "raw_text": text
    }
    return row


# ---------- 메인 파이프라인 ----------

def collect_yf_links(driver):
    print("[INFO] 검색 페이지 접속")
    driver.get(SEARCH_URL)
    time.sleep(3)

    scroll_until_stable(driver, pause=2.0, max_rounds=20)
    urls = collect_detail_links(driver)

    # YF 쏘나타 관련 링크만 1차 필터
    print(f"[INFO] 상세 링크 총 {len(urls)}건 수집")
    return urls


def scrape_detail_pages(driver, urls, max_items=None):
    rows = []

    target_urls = urls[:max_items] if max_items else urls

    for idx, url in enumerate(target_urls, start=1):
        try:
            car_id = extract_car_id(url)
            print(f"[INFO] ({idx}/{len(target_urls)}) 수집 중: {car_id}")

            driver.get(url)
            time.sleep(2.5)

            body_text = driver.find_element(By.TAG_NAME, "body").text
            text = normalize_text(body_text)

            row = parse_detail_text(text, url)
            rows.append(row)

        except Exception as e:
            print(f"[ERROR] {url} 수집 실패: {e}")

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.drop_duplicates(subset=["매물ID"]).reset_index(drop=True)

    return df


def clean_dataframe(df: pd.DataFrame):
    if df.empty:
        return df

    # YF 쏘나타가 아닌 것 제거
    df = df[df["차량명"].fillna("").str.contains("YF 쏘나타", na=False)].copy()

    # 가격 / 연식 / 주행거리 없는 행은 우선 제거
    df = df.dropna(subset=["가격_만원", "연식", "주행거리_km"])

    # 이상치 아주 거칠게 정리
    df = df[(df["가격_만원"] > 0) & (df["가격_만원"] < 5000)]
    df = df[(df["주행거리_km"] >= 0) & (df["주행거리_km"] < 500000)]
    df = df[(df["연식"] >= 2009) & (df["연식"] <= 2014)]

    df = df.reset_index(drop=True)
    return df


def main():
    driver = setup_driver(headless=False)

    try:
        detail_urls = collect_yf_links(driver)

        # 처음엔 10~15개만 시험해보는 게 좋아
        df_raw = scrape_detail_pages(driver, detail_urls, max_items=15)

        print("\n[INFO] 원본 추출 결과")
        print(df_raw.head())

        df_clean = clean_dataframe(df_raw)

        print("\n[INFO] 정리 후 결과")
        print(df_clean.head())
        print(f"\n총 수집 건수: {len(df_clean)}")

        df_raw.to_csv("encar_yf_sonata_detail_raw.csv", index=False, encoding="utf-8-sig")
        df_clean.to_csv("encar_yf_sonata_detail_clean.csv", index=False, encoding="utf-8-sig")

        print("[INFO] 저장 완료:")
        print("- encar_yf_sonata_detail_raw.csv")
        print("- encar_yf_sonata_detail_clean.csv")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] 검색 페이지 접속
[SCROLL] 1회차: 71436 -> 71436
[SCROLL] 2회차: 71436 -> 71436
[INFO] 스크롤 로딩 종료로 판단
[INFO] 상세 링크 총 210건 수집
[INFO] (1/15) 수집 중: 41054615
[INFO] (2/15) 수집 중: 39486507
[INFO] (3/15) 수집 중: 40903390
[INFO] (4/15) 수집 중: 41225357
[INFO] (5/15) 수집 중: 40477859
[INFO] (6/15) 수집 중: 41509017
[INFO] (7/15) 수집 중: 40955073
[INFO] (8/15) 수집 중: 41310694
[INFO] (9/15) 수집 중: 40955073
[INFO] (10/15) 수집 중: 41265617
[INFO] (11/15) 수집 중: 41019165
[INFO] (12/15) 수집 중: 41499870
[INFO] (13/15) 수집 중: 41054615
[INFO] (14/15) 수집 중: 41038293
[INFO] (15/15) 수집 중: 41018137

[INFO] 원본 추출 결과
       매물ID 제조사      모델               세부트림                      차량명   연식_원문  \
0  41054615  현대  YF 쏘나타        CVVL 럭셔리 연식        YF 쏘나타CVVL 럭셔리 연식  12/04식   
1  39486507  현대  YF 쏘나타           LPI 탑 연식           YF 쏘나타LPI 탑 연식  10/11식   
2  40903390  현대  YF 쏘나타  LPI 프리미어(장애인용) 연식  YF 쏘나타LPI 프리미어(장애인용) 연식  10/01식   
3  41225357  현대  YF 쏘나타         Y20 프라임 연식         YF 쏘나타Y20 프라임 연식  10/09식   
4  40477859  현대  YF 쏘나타  LPI

In [7]:
import pandas as pd
import re

df = pd.read_csv("encar_yf_sonata_detail_clean.csv")

def clean_name(name):
    if pd.isna(name):
        return None
    name = str(name)
    name = re.sub(r'\s*연식$', '', name).strip()
    name = re.sub(r'YF\s*쏘나타', 'YF 쏘나타 ', name).strip()
    name = re.sub(r'\s+', ' ', name)
    return name

def split_trim(name):
    if pd.isna(name):
        return None
    trim = str(name).replace("YF 쏘나타", "").strip()
    return trim if trim else None

def normalize_color(color):
    if pd.isna(color):
        return None
    color = str(color).strip()
    mapping = {
        "화이트": "흰색",
        "블랙": "검정",
        "실버": "은색",
        "그레이": "회색"
    }
    return mapping.get(color, color)

def normalize_accident(val):
    if pd.isna(val):
        return None
    val = str(val).strip()
    if val == "무사고":
        return 0
    if "사고" in val or "수리" in val:
        return 1
    return None

df["차량명"] = df["차량명"].apply(clean_name)
df["세부트림"] = df["차량명"].apply(split_trim)
df["색상"] = df["색상"].apply(normalize_color)
df["사고여부_flag"] = df["사고유무"].apply(normalize_accident)

# 옵션 중복 정리
df["옵션원문"] = df["옵션원문"].astype(str).str.replace("내비,네비", "내비", regex=False)
df["옵션원문"] = df["옵션원문"].str.replace("네비,내비", "내비", regex=False)

# 기본 확인
print(df[["차량명", "세부트림", "색상", "사고유무", "사고여부_flag"]].head(10))

df.to_csv("encar_yf_sonata_detail_refined.csv", index=False, encoding="utf-8-sig")
print("저장 완료: encar_yf_sonata_detail_refined.csv")

                     차량명            세부트림    색상        사고유무  사고여부_flag
0        YF 쏘나타 CVVL 럭셔리        CVVL 럭셔리  None         무사고          0
1           YF 쏘나타 LPI 탑           LPI 탑  None  사고/수리이력 의심          1
2  YF 쏘나타 LPI 프리미어(장애인용)  LPI 프리미어(장애인용)  None         무사고          0
3         YF 쏘나타 Y20 프라임         Y20 프라임  None  사고/수리이력 의심          1
4  YF 쏘나타 LPI 프리미어(장애인용)  LPI 프리미어(장애인용)    은색         무사고          0
5       YF 쏘나타 Y20 프라임블랙       Y20 프라임블랙    흰색         무사고          0
6      YF 쏘나타 Y20 프라임고급형      Y20 프라임고급형  None  사고/수리이력 의심          1
7         YF 쏘나타 LPI 럭셔리         LPI 럭셔리  None  사고/수리이력 의심          1
8         YF 쏘나타 Y20 럭셔리         Y20 럭셔리  None  사고/수리이력 의심          1
9       YF 쏘나타 Y20 프라임블랙       Y20 프라임블랙    검정         무사고          0
저장 완료: encar_yf_sonata_detail_refined.csv


In [9]:
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "LPG(일반인 구입)", "가솔린+전기", "하이브리드", "가솔린", "디젤", "LPG", "전기", "수소", "CNG"
]

TRANSMISSIONS = [
    "오토", "자동", "수동", "세미오토", "CVT"
]

COLOR_MAP = {
    "화이트": "흰색",
    "흰색": "흰색",
    "블랙": "검정",
    "검정": "검정",
    "검은색": "검정",
    "실버": "은색",
    "은색": "은색",
    "그레이": "회색",
    "회색": "회색",
    "쥐색": "회색",
    "청색": "파랑",
    "파랑": "파랑",
    "블루": "파랑",
    "빨강": "빨강",
    "레드": "빨강",
    "진주색": "진주",
    "진주": "진주",
    "베이지": "베이지",
    "브라운": "갈색",
    "갈색": "갈색"
}


# ---------------------------
# 드라이버
# ---------------------------

def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


# ---------------------------
# 유틸
# ---------------------------

def normalize_text(text):
    if text is None:
        return None
    return re.sub(r"\s+", " ", str(text)).strip()


def extract_car_id(url):
    if not url:
        return None
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def safe_text(element):
    try:
        return normalize_text(element.text)
    except:
        return None


def unique_join(text_list):
    cleaned = []
    seen = set()

    for t in text_list:
        t = normalize_text(t)
        if not t:
            continue
        if t not in seen:
            cleaned.append(t)
            seen.add(t)

    return " || ".join(cleaned) if cleaned else None


# ---------------------------
# 목록 페이지
# ---------------------------

def scroll_until_stable(driver, pause=2.0, max_rounds=20):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(f"[SCROLL] {i+1}회차: {last_height} -> {new_height}")

        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            print("[INFO] 스크롤 종료로 판단")
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except:
            pass

    urls = list(dict.fromkeys(urls))
    return urls


def collect_yf_links(driver):
    print("[INFO] 검색 페이지 접속")
    driver.get(SEARCH_URL)
    time.sleep(3)

    scroll_until_stable(driver, pause=2.0, max_rounds=20)
    urls = collect_detail_links(driver)

    print(f"[INFO] 상세 링크 {len(urls)}건 수집")
    return urls


# ---------------------------
# 상세페이지 섹션 수집
# ---------------------------

def collect_candidate_sections(driver):
    """
    페이지 구조가 바뀌어도 최대한 버티도록
    헤더/제목/테이블/리스트 기반 후보를 넓게 수집한다.
    """
    sections = {
        "기본정보_text": [],
        "차량상태_text": [],
        "옵션_text": [],
        "설명_text": []
    }

    all_blocks = driver.find_elements(By.CSS_SELECTOR, "section, article, div, ul, table")

    for block in all_blocks:
        txt = safe_text(block)
        if not txt or len(txt) < 20:
            continue

        low = txt.lower()

        # 기본정보 후보
        if any(k in txt for k in ["연식", "주행거리", "연료", "변속기", "배기량", "색상", "지역", "차종"]):
            sections["기본정보_text"].append(txt)

        # 차량상태 후보
        if any(k in txt for k in ["무사고", "사고", "교환", "판금", "수리", "침수", "용도", "렌트", "법인", "영업용", "소유자"]):
            sections["차량상태_text"].append(txt)

        # 옵션 후보
        if any(k in txt for k in ["옵션", "선루프", "내비", "네비", "스마트키", "열선", "통풍", "후방카메라", "가죽시트"]):
            sections["옵션_text"].append(txt)

        # 설명 후보
        if any(k in txt for k in ["차량설명", "판매자", "딜러", "특이사항", "강조", "장점"]):
            sections["설명_text"].append(txt)

    # body 전체도 보존
    sections["기본정보_text"] = unique_join(sections["기본정보_text"])
    sections["차량상태_text"] = unique_join(sections["차량상태_text"])
    sections["옵션_text"] = unique_join(sections["옵션_text"])
    sections["설명_text"] = unique_join(sections["설명_text"])

    return sections


# ---------------------------
# 라벨 기반 추출
# ---------------------------

def find_label_value(text, labels, value_pattern=r"([^\|\n\r]+)"):
    if not text:
        return None

    for label in labels:
        patterns = [
            rf"{label}\s*[:：]?\s*{value_pattern}",
            rf"{label}\s+{value_pattern}",
        ]
        for p in patterns:
            m = re.search(p, text, re.IGNORECASE)
            if m:
                return normalize_text(m.group(1))
    return None


def extract_name(text):
    if not text:
        return None

    m = re.search(r"(YF\s*쏘나타.*?)(?=\s+\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    if m:
        name = normalize_text(m.group(1))
        name = re.sub(r"\s*연식$", "", name).strip()
        return name

    return None


def split_model_trim(name):
    if not name:
        return None, None, None

    manufacturer = "현대"
    model = "YF 쏘나타"
    trim = normalize_text(name.replace("YF 쏘나타", "")) or None

    return manufacturer, model, trim


def extract_year_text(text):
    if not text:
        return None
    m = re.search(r"(\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    return m.group(1) if m else None


def convert_year_text(year_text):
    if not year_text:
        return None
    m = re.search(r"(\d{2})/\d{2}식", year_text)
    if not m:
        return None
    yy = int(m.group(1))
    return 2000 + yy


def extract_mileage(text):
    if not text:
        return None
    m = re.search(r"(\d[\d,]*)\s*km", text, re.IGNORECASE)
    return int(m.group(1).replace(",", "")) if m else None


def extract_price(text):
    if not text:
        return None
    m = re.search(r"(\d[\d,]*)\s*만원", text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_fuel(text):
    if not text:
        return None
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text):
    if not text:
        return None
    for region in REGIONS:
        if region in text:
            return region
    return None


def extract_transmission(text):
    if not text:
        return None

    label_value = find_label_value(text, ["변속기", "미션"], value_pattern=r"([가-힣A-Za-z]+)")
    if label_value:
        for tm in TRANSMISSIONS:
            if tm in label_value:
                return tm

    for tm in TRANSMISSIONS:
        if tm in text:
            return tm

    return None


def extract_displacement_cc(text):
    if not text:
        return None

    label_value = find_label_value(text, ["배기량"], value_pattern=r"(\d{3,4})\s*cc?")
    if label_value:
        m = re.search(r"(\d{3,4})", label_value)
        if m:
            return int(m.group(1))

    patterns = [
        r"(\d{3,4})\s*cc",
        r"배기량\s*(\d{3,4})"
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return int(m.group(1))

    return None


def normalize_color(color):
    if not color:
        return None
    color = normalize_text(color)
    return COLOR_MAP.get(color, color)


def extract_color(text):
    if not text:
        return None

    label_value = find_label_value(
        text,
        ["색상", "외장색", "외장 컬러"],
        value_pattern=r"([가-힣A-Za-z]+)"
    )
    if label_value:
        return normalize_color(label_value)

    for k, v in COLOR_MAP.items():
        if k in text:
            return v

    return None


def extract_accident_flag(condition_text):
    if not condition_text:
        return None

    t = normalize_text(condition_text)

    # 무사고 우선
    if any(k in t for k in ["완전무사고", "무사고"]):
        return "무사고"

    # 보다 보수적
    strong_signals = [
        "사고이력 있음",
        "교환",
        "판금",
        "수리이력",
        "수리 흔적"
    ]
    if any(k in t for k in strong_signals):
        return "사고이력"

    return None


def extract_flood_flag(condition_text):
    if not condition_text:
        return None

    t = normalize_text(condition_text)

    if "침수이력 없음" in t or "침수 없음" in t:
        return "침수없음"

    if "침수이력 있음" in t or "침수 차량" in t:
        return "침수이력"

    return None


def extract_usage_history(condition_text):
    if not condition_text:
        return None

    t = normalize_text(condition_text)
    found = []

    if "렌트" in t or "렌터카" in t:
        found.append("렌트이력")
    if "영업용" in t:
        found.append("영업용이력")
    if "법인" in t:
        found.append("법인이력")

    return ",".join(found) if found else None


def extract_owner_change_count(condition_text):
    if not condition_text:
        return None

    patterns = [
        r"소유자 변경\s*(\d+)\s*회",
        r"소유자변경\s*(\d+)\s*회",
        r"명의 변경\s*(\d+)\s*회"
    ]
    for p in patterns:
        m = re.search(p, condition_text)
        if m:
            return int(m.group(1))
    return None


def extract_options(option_text):
    if not option_text:
        return None

    option_keywords = [
        "선루프", "내비", "네비", "스마트키", "후방카메라",
        "열선시트", "통풍시트", "가죽시트", "메모리시트",
        "크루즈컨트롤", "스마트크루즈", "차선이탈", "어라운드뷰", "블랙박스"
    ]
    found = []

    for kw in option_keywords:
        if kw in option_text:
            found.append(kw)

    # 내비/네비 통합
    normalized = []
    for x in found:
        if x == "네비":
            x = "내비"
        normalized.append(x)

    normalized = sorted(set(normalized))
    return ",".join(normalized) if normalized else None


# ---------------------------
# 상세페이지 파싱
# ---------------------------

def parse_detail_page(driver, url):
    driver.get(url)
    time.sleep(2.5)

    raw_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
    sections = collect_candidate_sections(driver)

    merged_basic = " ".join(filter(None, [sections["기본정보_text"], raw_text]))
    merged_condition = " ".join(filter(None, [sections["차량상태_text"], raw_text]))
    merged_option = sections["옵션_text"]
    merged_desc = sections["설명_text"]

    name = extract_name(raw_text)
    manufacturer, model, trim = split_model_trim(name)
    year_text = extract_year_text(raw_text)

    row = {
        "매물ID": extract_car_id(url),
        "차량명": name,
        "제조사": manufacturer,
        "모델": model,
        "세부트림": trim,
        "연식_원문": year_text,
        "연식": convert_year_text(year_text),
        "주행거리_km": extract_mileage(merged_basic),
        "연료": extract_fuel(merged_basic),
        "변속기": extract_transmission(merged_basic),
        "배기량_cc": extract_displacement_cc(merged_basic),
        "색상": extract_color(merged_basic),
        "지역": extract_region(merged_basic),
        "가격_만원": extract_price(raw_text),
        "사고유무": extract_accident_flag(merged_condition),
        "침수유무": extract_flood_flag(merged_condition),
        "용도이력": extract_usage_history(merged_condition),
        "소유자변경횟수": extract_owner_change_count(merged_condition),
        "옵션원문": extract_options(merged_option),
        "기본정보_text": sections["기본정보_text"],
        "차량상태_text": sections["차량상태_text"],
        "옵션_text": sections["옵션_text"],
        "설명_text": sections["설명_text"],
        "raw_text": raw_text,
        "상세링크": url
    }

    return row


# ---------------------------
# 후처리
# ---------------------------

def clean_dataframe(df):
    if df.empty:
        return df

    df = df.copy()

    df = df[df["차량명"].fillna("").str.contains("YF 쏘나타", na=False)]
    df = df.drop_duplicates(subset=["매물ID"]).reset_index(drop=True)

    # 기본 필드 필터
    df = df.dropna(subset=["가격_만원", "연식", "주행거리_km"])

    df = df[(df["가격_만원"] > 0) & (df["가격_만원"] < 5000)]
    df = df[(df["주행거리_km"] >= 0) & (df["주행거리_km"] < 500000)]
    df = df[(df["연식"] >= 2009) & (df["연식"] <= 2014)]

    # 색상 정규화
    df["색상"] = df["색상"].apply(normalize_color)

    return df.reset_index(drop=True)


# ---------------------------
# 메인
# ---------------------------

def main():
    driver = setup_driver(headless=False)

    try:
        urls = collect_yf_links(driver)

        # 처음에는 5~10개만 테스트 추천
        target_urls = urls[:8]
        print(f"[INFO] 테스트 대상: {len(target_urls)}건")

        rows = []
        for i, url in enumerate(target_urls, start=1):
            try:
                print(f"[INFO] ({i}/{len(target_urls)}) 수집 중: {extract_car_id(url)}")
                row = parse_detail_page(driver, url)
                rows.append(row)
            except Exception as e:
                print(f"[ERROR] {url} 실패: {e}")

        df_raw = pd.DataFrame(rows)
        df_clean = clean_dataframe(df_raw)

        print("\n[RAW HEAD]")
        print(df_raw.head())

        print("\n[CLEAN HEAD]")
        print(df_clean.head())

        print(f"\n원본 건수: {len(df_raw)}")
        print(f"정제 건수: {len(df_clean)}")

        df_raw.to_csv("encar_yf_sonata_section_raw.csv", index=False, encoding="utf-8-sig")
        df_clean.to_csv("encar_yf_sonata_section_clean.csv", index=False, encoding="utf-8-sig")

        print("\n[INFO] 저장 완료")
        print("- encar_yf_sonata_section_raw.csv")
        print("- encar_yf_sonata_section_clean.csv")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] 검색 페이지 접속
[SCROLL] 1회차: 71502 -> 71502
[SCROLL] 2회차: 71502 -> 71502
[INFO] 스크롤 종료로 판단
[INFO] 상세 링크 210건 수집
[INFO] 테스트 대상: 8건
[INFO] (1/8) 수집 중: 40903390
[INFO] (2/8) 수집 중: 41509017
[INFO] (3/8) 수집 중: 39486507
[INFO] (4/8) 수집 중: 41225357
[INFO] (5/8) 수집 중: 41054615
[INFO] (6/8) 수집 중: 40955073
[INFO] (7/8) 수집 중: 40477859
[INFO] (8/8) 수집 중: 41265617

[RAW HEAD]
       매물ID                   차량명 제조사      모델            세부트림   연식_원문    연식  \
0  40903390  YF 쏘나타LPI 프리미어(장애인용)  현대  YF 쏘나타  LPI 프리미어(장애인용)  10/01식  2010   
1  41509017       YF 쏘나타Y20 프라임블랙  현대  YF 쏘나타       Y20 프라임블랙  10/10식  2010   
2  39486507           YF 쏘나타LPI 탑  현대  YF 쏘나타           LPI 탑  10/11식  2010   
3  41225357         YF 쏘나타Y20 프라임  현대  YF 쏘나타         Y20 프라임  10/09식  2010   
4  41054615        YF 쏘나타CVVL 럭셔리  현대  YF 쏘나타        CVVL 럭셔리  12/04식  2012   

   주행거리_km           연료 변속기  ...  침수유무  용도이력 소유자변경횟수  \
0   180575  LPG(일반인 구입)  자동  ...  None  None    None   
1   113663          가솔린  오토  ...  None  None 

In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

url = "https://fem.encar.com/cars/detail/40124634?advClickPosition=mweb_mhightlight_g88_t400&listAdvType=mhighlight&type=detail&view_type=hs_ad"

options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

driver.get(url)
time.sleep(3)

html = driver.page_source

with open("encar_detail_sample.html", "w", encoding="utf-8") as f:
    f.write(html)

print("저장 완료: encar_detail_sample.html")

driver.quit()

저장 완료: encar_detail_sample.html
